<a href="https://colab.research.google.com/github/eddiejaques/ml-code-samples/blob/main/Security%20Breaches%20Dataset/data_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Security Data — Logistic Regression (Multiple Logistic Regression)

## Dataset
Data sourced from `security_data_10000.csv` (synthetic data generated from original class dataset).

| Column | Description |
|--------|-------------|
| Sector | Industry sector of the company |
| CEO_Gender | Gender of the CEO (Male/Female) |
| Size | Company size (Large, Medium, Small) |
| Security_Invest | Security investment amount |
| Security_Breach_Att | Number of security breach attempts |
| Succ_Sec_Breaches | Number of successful security breaches |
| Sec_Rating | Overall security rating (High/Medium/Low) |
| CEO_Sec_Exp | CEO's level of security experience (High/Medium/Low) |
| LOT_in_Business | Length of time in business (years) |
| Stock_Market | Whether the company is listed on the stock market (Yes/No) |

---

## Objective
Predict whether a company is **listed on the stock market (Yes/No)** using all available features.

**Target variable:** `Stock_Market` (Yes = True / No = False)

---

## Data Analysis Plan

1.  **Load Data** — Read CSV into DataFrame
2.  **Train-Test Split** — Split before any feature engineering
3.  **Create Binary Label** — Encode `Stock_Market` as True/False
4.  **Drop Target from Features** — Remove `Stock_Market` from input features
5.  **Encode Categorical Features** — One-hot encode remaining categorical columns
6.  **Fit Logistic Regression** — Use `class_weight='balanced'` to handle class imbalance
7.  **Evaluate** — Confusion matrix and classification report on train and test

## Step 1 — Load Dataset

In [470]:
#LOAD THE DATASET


import pandas as pd

security_df = pd.read_csv("https://raw.githubusercontent.com/eddiejaques/ml-code-samples/refs/heads/main/Security%20Breaches%20Dataset/security-data.csv")

security_df.head()

,Sector,CEO_Gender,Size,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,Sec_Rating,CEO_Sec_Exp,LOT_in_Business,Stock_Market
0,Banking,Female,Small,17,11,2,Medium,High,13,No
1,Banking,Male,Small,18,12,4,High,Low,9,No
2,Banking,Male,Small,17,12,4,High,Medium,22,No
3,Banking,Male,Small,24,13,1,High,Medium,3,Yes
4,Banking,Male,Small,32,14,3,High,Medium,4,Yes


## Step 2 — Train-Test Split
Split performed **before** any feature engineering to prevent data leakage.

In [471]:
from sklearn.model_selection import train_test_split
security_df_train, security_df_test = train_test_split(security_df, test_size=0.2, random_state=42, stratify=security_df['Stock_Market'])

### Step 3 — Exploratory Data Analysis (EDA)

In [472]:
security_df_train.describe()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
count,48.000000,48.000000,48.000000,48.000000
mean,65.291667,58.041667,23.708333,8.666667
std,70.610999,70.441452,25.076019,6.008269
min,12.000000,10.000000,0.000000,1.000000
25%,20.500000,14.000000,3.000000,4.000000
50%,38.500000,35.500000,18.500000,8.000000
75%,92.750000,77.000000,34.000000,12.250000
max,435.000000,321.000000,100.000000,22.000000


In [473]:
security_df_train.isna().sum()

,0
Sector,0
CEO_Gender,0
Size,0
Security_Invest,0
Security_Breach_Att,0
Succ_Sec_Breaches,0
Sec_Rating,0
CEO_Sec_Exp,0
LOT_in_Business,0
Stock_Market,0


## Step 4 — Create Binary Label
Encode `Stock_Market` as a boolean: `True` = Yes, `False` = No.
This will be the target variable for the logistic regression model.

In [474]:
security_df_train_label = (security_df_train["Stock_Market"] == 'Yes')

security_df_test_label = (security_df_test["Stock_Market"] =='Yes')

In [475]:
security_df_train.shape

(48, 10)

## Step 5 — Drop Target Column from Features
`Stock_Market` must be removed from the input features — it is the target variable, not a predictor.

In [476]:
security_df_train = security_df_train.drop("Stock_Market", axis=1)
security_df_train.shape

(48, 9)

In [477]:
security_df_test = security_df_test.drop("Stock_Market", axis=1)
security_df_test.shape

(12, 9)

In [478]:
security_df_train.describe()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
count,48.000000,48.000000,48.000000,48.000000
mean,65.291667,58.041667,23.708333,8.666667
std,70.610999,70.441452,25.076019,6.008269
min,12.000000,10.000000,0.000000,1.000000
25%,20.500000,14.000000,3.000000,4.000000
50%,38.500000,35.500000,18.500000,8.000000
75%,92.750000,77.000000,34.000000,12.250000
max,435.000000,321.000000,100.000000,22.000000


In [489]:
numerical_cols = ["Security_Invest", "Security_Breach_Att", "Succ_Sec_Breaches", "LOT_in_Business"]
categorical_cols = ["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "CEO_Gender"]

security_df_train_cat = pd.get_dummies(security_df_train[categorical_cols], dtype=int)
security_df_test_cat = pd.get_dummies(security_df_test[categorical_cols], dtype=int)

# Align columns after one-hot encoding
aligned_cols = list(set(security_df_train_cat.columns) | set(security_df_test_cat.columns))
security_df_train_cat = security_df_train_cat.reindex(columns=aligned_cols, fill_value=0)
security_df_test_cat = security_df_test_cat.reindex(columns=aligned_cols, fill_value=0)

security_df_train_final = pd.concat([security_df_train[numerical_cols].reset_index(drop=True), security_df_train_cat.reset_index(drop=True)], axis=1)
security_df_test_final = pd.concat([security_df_test[numerical_cols].reset_index(drop=True), security_df_test_cat.reset_index(drop=True)], axis=1)


In [485]:
security_df_train_final.head()

,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,CEO_Gender_Female,CEO_Gender_Male,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.482433,3.772509,3.074611,-0.784927
1,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,-0.676837,-0.144062,-0.109148,0.056066
2,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,-0.676837,-0.531415,-0.834561,0.056066
3,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.390597,-0.474029,-0.391253,2.242648
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,-0.762709,-0.545762,-0.915163,0.897059


## Step 6 — Encode Categorical Features
One-hot encode all remaining categorical columns. `dtype=int` ensures 0/1 output instead of True/False.
Original categorical columns are dropped after encoding to avoid duplication.

In [480]:
security_df_train_final.head()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,CEO_Gender_Female,CEO_Gender_Male
30,99,321,100,4,0,0,1,0,1,0,0,0,1,1,0,0,0,1
38,18,48,21,9,0,1,0,0,1,0,0,0,1,1,0,0,0,1
20,18,21,3,9,1,0,0,0,1,0,1,0,0,1,0,0,1,0
58,38,25,14,22,0,1,0,1,0,0,1,0,0,0,1,0,0,1
22,12,20,1,14,1,0,0,0,1,0,0,1,0,0,0,1,1,0


In [481]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

security_df_train_final = pd.DataFrame(imputer.fit_transform(security_df_train_final), columns=security_df_train_final.columns)

security_df_test_final = pd.DataFrame(imputer.transform(security_df_test_final), columns=security_df_test_final.columns)

security_df_train_final.head()


,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,CEO_Gender_Female,CEO_Gender_Male
0,99.0,321.0,100.0,4.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,18.0,48.0,21.0,9.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
2,18.0,21.0,3.0,9.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,38.0,25.0,14.0,22.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,12.0,20.0,1.0,14.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [486]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numerical_cols = ["Security_Invest", "Security_Breach_Att", "Succ_Sec_Breaches", "LOT_in_Business"]

security_df_train_scaled = pd.DataFrame(scaler.fit_transform(security_df_train_final[numerical_cols]), columns=numerical_cols)

security_df_test_scaled = pd.DataFrame(scaler.transform(security_df_test_final[numerical_cols]), columns=numerical_cols)

security_df_train_final = pd.concat([security_df_train_final.drop(columns=numerical_cols), security_df_train_scaled], axis=1)
security_df_test_final =  pd.concat([security_df_test_final.drop(columns=numerical_cols), security_df_test_scaled], axis=1)

security_df_train_final.head()
security_df_test_final.head()

,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,CEO_Gender_Female,CEO_Gender_Male,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,-0.476469,-0.560108,-0.794261,1.569854
1,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.198031,-0.187101,-0.068847,0.728861
2,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.233166,-0.230140,0.092356,-0.280331
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,-0.590965,-0.474029,-0.391253,-0.953125
4,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,-0.018486,-0.187101,-0.189749,2.074449


## Step 7 — Fit Logistic Regression
`class_weight='balanced'` corrects for class imbalance (more Male than Female CEOs in the dataset).
Without this, the model defaults to predicting the majority class only.

In [487]:
from sklearn.linear_model import LogisticRegression

logistic_model = log_reg = LogisticRegression(class_weight="balanced", solver='liblinear', random_state=42)

logistic_model.fit(security_df_train_final, security_df_train_label)

security_df_train_label_pred = logistic_model.predict(security_df_train_final)

security_df_test_label_pred = logistic_model.predict(security_df_test_final)
print(f"Shape of security_df_test_final: {security_df_test_final.shape}")
print(f"Shape of security_df_test_label_pred: {security_df_test_label_pred.shape}")

Shape of security_df_test_final: (12, 18)
Shape of security_df_test_label_pred: (12,)


In [488]:
from sklearn.metrics import confusion_matrix, classification_report
print("Confusion Matrix - Train:")

print(confusion_matrix(security_df_train_label, security_df_train_label_pred))

print("Classification Report - Train:")

print(classification_report(security_df_train_label, security_df_train_label_pred))

print("Confusion Matrix - Test:")

print(confusion_matrix(security_df_test_label, security_df_test_label_pred))

print("Classification Report - Test:")

print(classification_report(security_df_test_label, security_df_test_label_pred))

Confusion Matrix - Train:
[[14  0]
 [ 7 27]]
Classification Report - Train:
              precision    recall  f1-score   support

       False       0.67      1.00      0.80        14
        True       1.00      0.79      0.89        34

    accuracy                           0.85        48
   macro avg       0.83      0.90      0.84        48
weighted avg       0.90      0.85      0.86        48

Confusion Matrix - Test:
[[3 1]
 [4 4]]
Classification Report - Test:
              precision    recall  f1-score   support

       False       0.43      0.75      0.55         4
        True       0.80      0.50      0.62         8

    accuracy                           0.58        12
   macro avg       0.61      0.62      0.58        12
weighted avg       0.68      0.58      0.59        12



## Step 8 — Evaluate
- **Confusion matrix** — shows True/False Positives and Negatives
- **Precision** — of all predicted Male/Female, how many were correct
- **Recall** — of all actual Male/Female, how many were caught
- **F1-score** — harmonic mean of precision and recall

# Summary


We tried